# Clustering — DBSCAN
Algoritmo baseado em densidade: descobre clusters de forma arbitrária e identifica outliers (ruído).

In [1]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score, davies_bouldin_score

plt.rcParams['figure.figsize'] = (10, 5)
sns.set_theme(style='whitegrid')

X_scaled = np.load('../dataset/X_scaled.npy')

with open('../dataset/preprocessamento.pkl', 'rb') as f:
    meta = pickle.load(f)

df_orig = meta['df_original']
print(f'Matriz carregada: {X_scaled.shape}')

Matriz carregada: (2000, 40)


## 1. K-Distance Plot — escolha do eps
Ordena as distâncias ao k-ésimo vizinho mais próximo. O "joelho" indica o valor ideal de `eps`.

In [2]:
MIN_S = 5  # min_samples (ajuste aqui se necessário)

nbrs = NearestNeighbors(n_neighbors=MIN_S).fit(X_scaled)
distancias, _ = nbrs.kneighbors(X_scaled)
distancias_k = np.sort(distancias[:, -1])  # distância ao k-ésimo vizinho

plt.figure(figsize=(10, 4))
plt.plot(distancias_k, color='steelblue')
plt.title(f'K-Distance Plot (k={MIN_S}) — procure o joelho para definir eps')
plt.xlabel('Pontos ordenados')
plt.ylabel(f'Distância ao {MIN_S}º vizinho')
plt.tight_layout()
plt.show()

print(f'Percentis das distâncias:')
for p in [50, 75, 85, 90, 95]:
    print(f'  p{p}: {np.percentile(distancias_k, p):.3f}')

ValueError: Input X contains NaN.
NearestNeighbors does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

## 2. Busca de Parâmetros — grade eps × min_samples
Testa combinações e filtra as que geram pelo menos 2 clusters válidos.

In [ ]:
eps_values  = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0]
mins_values = [3, 5, 10, 15]

resultados = []

for eps in eps_values:
    for min_s in mins_values:
        db = DBSCAN(eps=eps, min_samples=min_s)
        lbs = db.fit_predict(X_scaled)
        n_clusters = len(set(lbs)) - (1 if -1 in lbs else 0)
        n_noise    = (lbs == -1).sum()
        pct_noise  = n_noise / len(lbs) * 100

        if n_clusters >= 2:
            # Silhouette só calculado sem ruído
            mask = lbs != -1
            sil = silhouette_score(X_scaled[mask], lbs[mask]) if mask.sum() > 1 else -1
            resultados.append({
                'eps': eps, 'min_samples': min_s,
                'n_clusters': n_clusters, 'n_noise': n_noise,
                'pct_noise': round(pct_noise, 1), 'silhouette': round(sil, 4)
            })

df_res = pd.DataFrame(resultados).sort_values('silhouette', ascending=False)
print(f'{len(df_res)} combinações válidas (≥2 clusters):')
display(df_res.head(10))

## 3. Treinar DBSCAN com parâmetros escolhidos
> **Ajuste** `EPS` e `MIN_S` com base na tabela acima.

In [ ]:
EPS   = 2.0  # ajuste aqui
MIN_S = 5    # ajuste aqui

dbscan = DBSCAN(eps=EPS, min_samples=MIN_S)
labels_db = dbscan.fit_predict(X_scaled)

n_clusters = len(set(labels_db)) - (1 if -1 in labels_db else 0)
n_noise    = (labels_db == -1).sum()

print(f'DBSCAN (eps={EPS}, min_samples={MIN_S})')
print(f'  Clusters encontrados: {n_clusters}')
print(f'  Pontos de ruído: {n_noise} ({n_noise/len(labels_db)*100:.1f}%)')
print(f'  Distribuição: {pd.Series(labels_db).value_counts().sort_index().to_dict()}')

mask_valid = labels_db != -1
if mask_valid.sum() > 1 and n_clusters >= 2:
    sil = silhouette_score(X_scaled[mask_valid], labels_db[mask_valid])
    dbi = davies_bouldin_score(X_scaled[mask_valid], labels_db[mask_valid])
    print(f'  Silhouette (sem ruído): {sil:.4f}')
    print(f'  Davies-Bouldin (sem ruído): {dbi:.4f}')
else:
    sil, dbi = None, None
    print('Não foi possível calcular métricas (clusters insuficientes).')

## 4. Visualização PCA (2D)

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

palette = plt.cm.tab10.colors
labels_unicas = sorted(set(labels_db))

plt.figure(figsize=(10, 6))
for lbl in labels_unicas:
    mask = labels_db == lbl
    nome = f'Ruído' if lbl == -1 else f'Cluster {lbl}'
    cor  = 'lightgray' if lbl == -1 else palette[lbl % len(palette)]
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1],
                c=[cor], label=nome, alpha=0.5 if lbl != -1 else 0.3, s=20)

plt.title(f'DBSCAN (eps={EPS}, min_samples={MIN_S}) — Projeção PCA 2D')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.legend()
plt.tight_layout()
plt.show()

## 5. Perfil dos clusters (excluindo ruído)

In [ ]:
df_perfil = df_orig.copy()
df_perfil['Cluster'] = labels_db

# Apenas pontos não-ruído
df_validos = df_perfil[df_perfil['Cluster'] != -1]

num_cols = ['Age', 'Years_Experience', 'Salary_Before_AI', 'Work_Hours_Per_Week', 'Job_Satisfaction']

print('=== MÉDIAS NUMÉRICAS POR CLUSTER ===')
display(df_validos.groupby('Cluster')[num_cols].mean().round(1))

print('\n=== JOB STATUS POR CLUSTER (%) ===')
status_pct = (df_validos.groupby(['Cluster', 'Job_Status'])
                         .size()
                         .unstack(fill_value=0)
                         .apply(lambda r: r / r.sum() * 100, axis=1)
                         .round(1))
display(status_pct)

## 6. Análise dos outliers (ruído)

In [ ]:
df_ruido = df_perfil[df_perfil['Cluster'] == -1]
print(f'Total de outliers: {len(df_ruido)} ({len(df_ruido)/len(df_perfil)*100:.1f}%)')

if len(df_ruido) > 0:
    print('\nDistribuição dos outliers por Job_Status:')
    print(df_ruido['Job_Status'].value_counts())
    print('\nDistribuição por Automation_Risk:')
    print(df_ruido['Automation_Risk'].value_counts())

## 7. Salvar labels para comparação final

In [ ]:
import json

np.save('../dataset/labels_dbscan.npy', labels_db)

metricas_db = {
    'silhouette': float(sil) if sil is not None else None,
    'davies_bouldin': float(dbi) if dbi is not None else None,
    'n_clusters': n_clusters,
    'n_noise': int(n_noise),
    'eps': EPS,
    'min_samples': MIN_S
}
with open('../dataset/metricas_dbscan.json', 'w') as f:
    json.dump(metricas_db, f)

print('Labels e métricas salvos.')